# Minkowski Structure Metrics

**Voronoi face-area weighted Steinhardt parameters**

Standard Steinhardt $q_l$ values weight all neighbors equally.  Minkowski structure metrics (Mickel *et al.*, 2013) replace the equal weighting with Voronoi face areas, making the descriptor continuous and independent of the neighbor-list definition.

$$
q_l^{\mathrm{Mink}}(i) = \sqrt{\frac{4\pi}{2l+1}
    \sum_{m=-l}^{l} \left| \sum_j \frac{A_j}{\sum_k A_k}
    Y_{lm}(\hat{\mathbf{r}}_{ij}) \right|^2}
$$

where $A_j$ is the Voronoi face area shared between atoms $i$ and $j$.

In [ ]:
import pyscal3
from pyscal3.structures import make_crystal
import numpy as np

## Perfect Crystals

For ideal lattices every atom has the same Voronoi cell, so the Minkowski $q_l$ is a single constant characteristic of the structure.

In [ ]:
structures = {
    "fcc": make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4)),
    "bcc": make_crystal("bcc", lattice_constant=2.87, repetitions=(4, 4, 4)),
    "hcp": make_crystal("hcp", lattice_constant=3.21, repetitions=(4, 4, 4)),
}

print("{:<10} {:>10} {:>10}".format("Structure", "q4_Mink", "q6_Mink"))
print("-" * 32)

for name, atoms in structures.items():
    q4, q6 = pyscal3.minkowski_parameter(atoms, l=[4, 6])
    print(f"{name:<10} {q4[0]:>10.4f} {q6[0]:>10.4f}")

## Comparison: Standard vs Minkowski $q_l$

Standard Steinhardt $q_l$ with a cutoff-based neighbor list yields different values depending on the cutoff.  Minkowski $q_l$ removes this dependence by weighting neighbors with Voronoi face areas.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4))

# Standard Steinhardt (adaptive cutoff)
pyscal3.find_neighbors(fcc, method="cutoff", cutoff=0)
[q6_std] = pyscal3.steinhardt_parameter(fcc, l=6)

# Minkowski (Voronoi-weighted)
[q6_mink] = pyscal3.minkowski_parameter(fcc, l=6)

print(f"Standard  q6 = {q6_std[0]:.6f}")
print(f"Minkowski q6 = {q6_mink[0]:.6f}")

## Sensitivity to Thermal Disorder

Adding random displacements breaks the Voronoi symmetry, spreading the per-atom $q_6^{\mathrm{Mink}}$ values.

In [ ]:
print("{:<8} {:>12} {:>12}".format("Noise", "q6 mean", "q6 std"))
print("-" * 34)

for noise in [0.0, 0.02, 0.05, 0.1, 0.2]:
    atoms = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4), noise=noise)
    [q6] = pyscal3.minkowski_parameter(atoms, l=6)
    print(f"{noise:<8.2f} {q6.mean():>12.6f} {q6.std():>12.6f}")

## Face-Area Exponent

The `voroexp` parameter controls how strongly the face area biases the weighting.  With $\alpha=0$ all neighbors are equally weighted (recovering the standard Steinhardt parameter with all Voronoi neighbors), while $\alpha=1$ (default) gives the standard Minkowski metric.

In [ ]:
fcc = make_crystal("fcc", lattice_constant=4.05, repetitions=(4, 4, 4))

for alpha in [0, 1, 2, 3]:
    [q6] = pyscal3.minkowski_parameter(fcc, l=6, voroexp=alpha)
    print(f"voroexp={alpha}  q6 = {q6[0]:.6f}")

## References

1. W. Mickel, S. C. Kapfer, G. E. Schröder-Turk and K. Mecke, "Shortcomings of the bond orientational order parameters for the analysis of disordered particulate matter", *J. Chem. Phys.* **138**, 044501 (2013). [doi:10.1063/1.4774084](https://doi.org/10.1063/1.4774084)

2. P. J. Steinhardt, D. R. Nelson and M. Ronchetti, "Bond-orientational order in liquids and glasses", *Phys. Rev. B* **28**, 784 (1983). [doi:10.1103/PhysRevB.28.784](https://doi.org/10.1103/PhysRevB.28.784)